# Court Document Pipeline (GPU — Colab, Apple Silicon, or RunPod)

Runs the **GPU vision** + **LLM label/feature** stages of the pipeline. Auto-detects the runtime:

- **Google Colab (CUDA)** — uses `Qwen/Qwen2-VL-7B-Instruct` with bitsandbytes 4-bit quantization on a T4. PDF downloads happen first on the laptop via `local_pdf_pipeline.ipynb` because Cloudflare blocks Colab IPs from `webapps.sftc.org`.
- **Apple Silicon Mac (MLX)** — uses `mlx-community/Qwen2-VL-7B-Instruct-4bit` via `mlx-vlm`. Native Metal acceleration, ~4 GB resident. Reads/writes the team's shared Drive folder via **Google Drive for Desktop** (mounted at `~/Library/CloudStorage/GoogleDrive-…/My Drive/litigation-pipeline`).
- **Generic CUDA host (e.g. RunPod, Lambda Labs)** — uses the same bitsandbytes path as Colab. Drive is mounted via `rclone` at `/workspace/drive/litigation-pipeline`. The "Setup rclone + mount Drive" cell handles install + mount; you must **upload your `rclone.conf` to `/workspace/rclone.conf` via Jupyter Lab's file browser** before running it.

### What this notebook does
1. **Find scanned PDFs** in the shared Drive folder that need vision-based OCR
2. **Extract text** with Qwen2-VL-7B on GPU (only for pages PyMuPDF couldn't read)
3. **Extract labels** (win/loss, amounts) from outcome documents via GPT-4o-mini
4. **Extract features** (~40 existence-based booleans per case) via GPT-4o-mini
5. **Bundle results** as a zip download (Colab only — Mac and RunPod outputs already live in synced Drive)

### Team workflow
All team members share the same Google Drive folder. Each person sets a different `MY_WORKER` / `TOTAL_WORKERS` in the path-setup cell to split feature-extraction work (Ernesto=0, Alexander=1, Borna=2; `TOTAL_WORKERS=3`).

### ⚠️ Never paste credentials into a notebook cell
The `rclone.conf` file contains a long-lived OAuth refresh token; the HuggingFace token grants HF account access. If pasted into a cell, both end up in the .ipynb source and get committed to git. Always use file upload (Jupyter Lab file browser) for the rclone conf, and `from huggingface_hub import login; login()` for HF if needed.

In [1]:
import platform, subprocess
print("Hostname:", platform.node())
print("OS:", platform.system(), platform.release())
print("Python at:", subprocess.run(["which", "python"], capture_output=True, text=True).stdout.strip())
print()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


Hostname: 25b3cb7f786b
OS: Linux 6.8.0-48-generic
Python at: /usr/local/bin/python

Tue May 12 06:45:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:01:00.0 Off |                  Off |
| 30%   29C    P8             23W /  230W |       2MiB /  24564MiB |      0%      Default |
|                                         |             

In [2]:
# Upload rclone.conf via masked input (no leak into notebook source).
# On Mac, run this first to get a base64 single-line copy of your rclone.conf:
#     base64 -i ~/.config/rclone/rclone.conf | pbcopy
# Then run this cell, paste into the hidden prompt that appears, hit Enter.
# Skipped on Mac/Colab — this only matters on RunPod where rclone.conf isn't already on disk.
import base64
import getpass
import platform
import sys
from pathlib import Path

_is_runpod = (
    platform.system() == "Linux"
    and "google.colab" not in sys.modules
)

if not _is_runpod:
    print("Skipping rclone.conf upload — not on a Linux CUDA host.")
else:
    conf_path = Path.home() / ".config/rclone/rclone.conf"
    if conf_path.exists():
        print(f"{conf_path} already exists — skipping upload.")
    else:
        b64 = getpass.getpass("Paste base64 of rclone.conf (input is hidden): ").strip()
        contents = base64.b64decode(b64).decode("utf-8")
        conf_path.parent.mkdir(parents=True, exist_ok=True)
        conf_path.write_text(contents)
        print(f"Wrote {conf_path} — {len(contents)} bytes")

Wrote /root/.config/rclone/rclone.conf — 536 bytes


In [3]:
# Generic CUDA / RunPod only — install rclone, verify config, sync Drive contents to local disk.
# RunPod containers can't mount FUSE (no privileged access), so we use rclone copy
# instead of rclone mount. Outputs are synced back to Drive in the post-OCR cell.
# Skipped on Mac (Drive Desktop already syncs) and on Colab (drive.mount in path-setup cell).
#
# Re-running this cell: if pdfs/ already has >100 files locally, the bulk sync is
# skipped (saves a 1-2 min rclone scan). To force a re-sync after teammates upload
# new PDFs, set FORCE_RESYNC = True below.
FORCE_RESYNC = False

import os
import platform
import subprocess
import sys
from pathlib import Path


def _is_runpod_like():
    if platform.system() != "Linux":
        return False
    if "google.colab" in sys.modules:
        return False
    try:
        subprocess.run(["nvidia-smi"], capture_output=True, check=True, timeout=5)
        return True
    except Exception:
        return False


def _have(cmd: str) -> bool:
    return subprocess.run(["which", cmd], capture_output=True).returncode == 0


if not _is_runpod_like():
    print("Skipping rclone setup — not on a generic Linux CUDA host.")
else:
    # 1) Install rclone if missing
    if not _have("rclone"):
        print("Installing rclone ...")
        subprocess.run(
            ["bash", "-c", "curl -fsSL https://rclone.org/install.sh | sudo bash"],
            check=True,
        )
    rclone_ver = subprocess.run(
        ["rclone", "version"], capture_output=True, text=True
    ).stdout.splitlines()[0]
    print(f"rclone: {rclone_ver}")

    # 2) Verify rclone.conf is in place — upload via Jupyter Lab file browser
    # to /workspace/rclone.conf BEFORE running this cell. Never paste the conf
    # contents into a cell — it contains a long-lived OAuth refresh token.
    conf_path = Path.home() / ".config/rclone/rclone.conf"
    staged_conf = Path("/workspace/rclone.conf")
    if staged_conf.exists() and not conf_path.exists():
        conf_path.parent.mkdir(parents=True, exist_ok=True)
        conf_path.write_text(staged_conf.read_text())
        staged_conf.unlink()
        print(f"Moved staged conf → {conf_path}")

    if not conf_path.exists():
        raise RuntimeError(
            f"\n{conf_path} not found.\n"
            "Upload your rclone.conf to /workspace/rclone.conf via Jupyter Lab's "
            "file browser (drag-and-drop), then re-run this cell."
        )

    # 3) Sync Drive → local /workspace/litigation-pipeline. Skip the rclone scan
    # entirely if we already have a populated local sync (saves 1-2 min per re-run).
    local_dir = Path("/workspace/litigation-pipeline")
    local_dir.mkdir(parents=True, exist_ok=True)
    local_pdfs = local_dir / "pdfs"
    local_extracted = local_dir / "extracted"
    local_pdfs.mkdir(parents=True, exist_ok=True)
    local_extracted.mkdir(parents=True, exist_ok=True)

    n_local_pdfs = sum(1 for _ in local_pdfs.iterdir())

    if n_local_pdfs > 100 and not FORCE_RESYNC:
        n_local_txt = sum(1 for _ in local_extracted.iterdir())
        print(
            f"\nLocal sync already populated — {n_local_pdfs:,} PDFs, "
            f"{n_local_txt:,} .txt files. Skipping rclone scan."
        )
        print("Set FORCE_RESYNC = True at top of cell to pull new files from Drive.")
    else:
        print("\nSyncing pdfs/ from Drive (one-time bulk download, ~10-30 min)")
        subprocess.run(
            [
                "rclone", "copy", "drive:litigation-pipeline/pdfs",
                str(local_pdfs),
                "--transfers", "16",
                "--checkers", "32",
                "--progress",
            ],
            check=True,
        )

        print("\nSyncing extracted/ from Drive (existing .txt — skip re-OCR)")
        subprocess.run(
            [
                "rclone", "copy", "drive:litigation-pipeline/extracted",
                str(local_extracted),
                "--transfers", "16",
                "--checkers", "32",
                "--progress",
            ],
            check=True,
        )

        n_local_pdfs = sum(1 for _ in local_pdfs.iterdir())
        n_local_txt = sum(1 for _ in local_extracted.iterdir())
        print(f"\nLocal sync done — {n_local_pdfs:,} PDFs, {n_local_txt:,} .txt files at {local_dir}")

rclone: rclone v1.58.1

Local sync already populated — 7,335 PDFs, 7,692 .txt files. Skipping rclone scan.
Set FORCE_RESYNC = True at top of cell to pull new files from Drive.


In [4]:
!pip install --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu124


Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp312-cp312-linux_x86_64.whl.metadata (28 kB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp312-cp312-linux_x86_64.whl.metadata (6.1 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/cu124/nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (24.6 MB)
  Using cached https://download.pytorch.org/whl/cu124/nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (883 kB)
  Using cached https://dow

In [5]:
import platform
import sys

IS_MAC_ARM = platform.system() == "Darwin" and platform.machine() == "arm64"
IS_COLAB = "google.colab" in sys.modules

# Generic CUDA = Linux + nvidia-smi available + not Colab (e.g. RunPod, Lambda Labs).
# Detect lazily so the subprocess call doesn't run on Mac.
IS_GENERIC_CUDA = False
if not IS_MAC_ARM and not IS_COLAB:
    try:
        import subprocess
        subprocess.run(["nvidia-smi"], capture_output=True, check=True, timeout=5)
        IS_GENERIC_CUDA = True
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        pass

if IS_MAC_ARM:
    !pip install -q pymupdf "transformers>=4.45" Pillow "mlx-vlm>=0.1.0"
    BACKEND = "mlx"
    print("Backend: MLX (Apple Silicon)")
elif IS_COLAB or IS_GENERIC_CUDA:
    !pip install -q pymupdf "transformers>=4.45" accelerate "bitsandbytes>=0.46.1" qwen-vl-utils Pillow

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available — check `nvidia-smi` and the GPU allocation.")

    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    BACKEND = "cuda"
    env_label = "Colab" if IS_COLAB else "Generic CUDA (e.g. RunPod)"
    print(f"Backend: CUDA — {gpu_name} ({gpu_mem:.1f} GB) [{env_label}]")
else:
    raise RuntimeError(
        "Unsupported environment. Need Apple Silicon (MLX), Colab CUDA, "
        "or a generic Linux CUDA host (RunPod / Lambda Labs / vast.ai)."
    )

Backend: CUDA — NVIDIA RTX A5000 (25.3 GB) [Generic CUDA (e.g. RunPod)]


## Setup

Clone the repo (gets the scraper code + `valid_cases.json`) and optionally
mount Google Drive so downloaded files persist between sessions.

In [6]:
import os
import sys
from pathlib import Path

# ============================================================
# CONFIGURATION — edit these if needed
# ============================================================
# TEAM WORK SPLITTING — divide cases across team members for feature extraction.
# Each person picks a unique MY_WORKER (0-indexed). Set TOTAL_WORKERS = 1 to do everything yourself.
# Defaults: Ernesto=0, Alexander=1, Borna=2; TOTAL_WORKERS=3
MY_WORKER = 0
TOTAL_WORKERS = 1
# ============================================================

if IS_MAC_ARM:
    # Local Mac — repo is already cloned, Drive is mounted by Drive Desktop.
    REPO_DIR = "/Users/ernesto/Desktop/MLOPS-Project/litigation-outcome-pipeline"
    DRIVE_DIR = (
        "/Users/ernesto/Library/CloudStorage/"
        "GoogleDrive-ernestod1998@gmail.com/My Drive/litigation-pipeline"
    )
    if not Path(DRIVE_DIR).exists():
        raise RuntimeError(
            f"Drive Desktop path not found: {DRIVE_DIR}\n"
            "Install Google Drive for Desktop and set the shared 'litigation-pipeline' "
            "folder to 'Available offline'."
        )
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

elif IS_GENERIC_CUDA:
    # RunPod / Lambda / vast.ai. Drive contents synced to local disk via rclone copy
    # (FUSE mount unavailable in non-privileged containers). Outputs sync back to
    # Drive in the "Sync results back" cell after Step 3.
    REPO_URL = "https://github.com/karimiborna/litigation-outcome-pipeline.git"
    BRANCH = "ernies_scraper"
    REPO_DIR = "/workspace/litigation-outcome-pipeline"
    DRIVE_DIR = "/workspace/litigation-pipeline"

    if not (Path(DRIVE_DIR) / "pdfs").exists():
        raise RuntimeError(
            f"Local Drive sync not found at {DRIVE_DIR}/pdfs\n"
            "Run the rclone sync cell above first."
        )

    if not Path(REPO_DIR).exists():
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        print(f"Repo already cloned at {REPO_DIR} — syncing to origin/{BRANCH}")
        !cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

    # Install ONLY the runtime deps the notebook actually imports.
    # Critically, do NOT let pip resolve the repo's full dependency tree on RunPod —
    # that pulls torch 2.11.0 fresh, which doesn't match the base image's torchvision
    # and breaks every time. The torch fix cell handles torch + torchvision separately.
    print("Installing notebook runtime deps (skipping torch — handled separately) ...")
    !pip install -q pydantic pydantic-settings openai pymupdf "transformers>=4.45" Pillow accelerate "bitsandbytes>=0.46.1" qwen-vl-utils httpx tiktoken aiohttp tenacity backoff
    # Install the package itself (so `from scraper.config import ...` works) WITHOUT touching deps.
    !pip install -q --no-deps -e {REPO_DIR}

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

else:
    # Colab — clone the repo, editable install, mount Drive.
    REPO_URL = "https://github.com/karimiborna/litigation-outcome-pipeline.git"
    BRANCH = "ernies_scraper"
    DRIVE_DIR = "/content/drive/MyDrive/litigation-pipeline"
    REPO_DIR = "/content/litigation-outcome-pipeline"

    if not Path(REPO_DIR).exists():
        !git config --global credential.helper ''
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        print(f"Repo already cloned at {REPO_DIR} — syncing to origin/{BRANCH}")
        !cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

    !pip install -q -e {REPO_DIR}

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # `files` is imported here so the Results cell at the bottom can call files.download().
    from google.colab import drive, files
    drive.mount("/content/drive")

# Common to all three — same DRIVE_DIR layout, different physical mount point
PDF_DIR = f"{DRIVE_DIR}/pdfs"
OUTPUT_DIR = f"{DRIVE_DIR}/extracted"
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not list(Path(PDF_DIR).glob("*.pdf")):
    raise RuntimeError(
        f"No PDFs found in {PDF_DIR}. Run notebooks/local_pdf_pipeline.ipynb first "
        "— Cloudflare blocks Colab from downloading directly from webapps.sftc.org."
    )

print(f"\nPDFs read from:  {PDF_DIR}")
print(f"Text written to: {OUTPUT_DIR}")
print(f"Worker:          {MY_WORKER}/{TOTAL_WORKERS}")

Repo already cloned at /workspace/litigation-outcome-pipeline — syncing to origin/ernies_scraper
From https://github.com/karimiborna/litigation-outcome-pipeline
 * branch            ernies_scraper -> FETCH_HEAD
HEAD is now at 533a5fd fix: restore v2 feature pipeline end-to-end
Installing notebook runtime deps (skipping torch — handled separately) ...

PDFs read from:  /workspace/litigation-pipeline/pdfs
Text written to: /workspace/litigation-pipeline/extracted
Worker:          0/1


## Step 1 — Find scanned PDFs that need GPU OCR

`local_pdf_pipeline.ipynb` already ran PyMuPDF on the laptop, writing `.txt` files for every PDF with a usable text layer. Whatever PDF doesn't have a `.txt` companion is a scanned image — those are the ones that need the GPU.

In [7]:
from scraper.config import is_doc_type_wanted as is_doc_wanted


def _doc_type(p):
    return p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem


all_pdfs = sorted(Path(PDF_DIR).glob("*.pdf"))
whitelisted = [p for p in all_pdfs if is_doc_wanted(_doc_type(p))]
needs_gpu = [p for p in whitelisted if not (Path(OUTPUT_DIR) / f"{p.stem}.txt").exists()]

print(f"PDFs in Drive:             {len(all_pdfs)}")
print(f"Whitelisted doc types:     {len(whitelisted)}")
print(f"Already extracted to .txt: {len(whitelisted) - len(needs_gpu)}")
print(f"Need GPU vision model:     {len(needs_gpu)}")

PDFs in Drive:             7335
Whitelisted doc types:     7220
Already extracted to .txt: 7220
Need GPU vision model:     0


## Step 2 — Load the vision model

Loads **Qwen2-VL-7B-Instruct** with 4-bit quantization (~4 GB VRAM). First run downloads ~4 GB of weights (cached for subsequent runs). Skipped entirely if Step 1 found nothing for the GPU.

In [19]:
# Fix torch / torchvision install on RunPod (only needed once per pod after the editable install).
# The repo's pip install -e pulls a torch version that doesn't match the base image's torchvision,
# leaving the env in a broken state where torch internals reference dtypes that don't exist
# (e.g. AttributeError: module 'torch' has no attribute 'float8_e8m0fnu').
#
# This cell wipes torch/torchvision/torchaudio/triton and reinstalls a matched cu124 set
# (compatible with NVIDIA driver 565+ that ships on most RunPod pods).
#
# AFTER RUNNING THIS CELL: restart the kernel (notebook toolbar → Restart) and re-run from cell 1.
#
# Idempotent: safe to skip on re-runs once torch is healthy. Set FIX_TORCH = True to run.
FIX_TORCH = True

if FIX_TORCH and IS_GENERIC_CUDA:
    import sys

    print("Wiping any existing torch/torchvision/torchaudio/triton ...")
    !pip uninstall -y -q torch torchvision torchaudio triton
    !pip cache purge

    # cu124 build works with NVIDIA driver 535+ (covers nearly all RunPod pods).
    # Don't use cu128 — that needs driver 570+ which most RunPod pods don't have.
    print("\nInstalling matched torch + torchvision against CUDA 12.4 ...")
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124

    # Drop cached torch modules so a re-import in the same session picks up the new install.
    # (Kernel restart is still required for full effect — some C extensions are loaded once.)
    for mod in list(sys.modules):
        if mod == "torch" or mod.startswith("torch."):
            del sys.modules[mod]

    print("\n" + "=" * 60)
    print("DONE. Now: notebook toolbar → Restart Kernel, then re-run from cell 1.")
    print("=" * 60)
else:
    print("Skipping torch fix (set FIX_TORCH = True or not on RunPod).")

In [20]:
model = None
processor = None
mlx_config = None

if not needs_gpu:
    print("All PDFs had selectable text — no model needed.")
elif BACKEND == "mlx":
    from mlx_vlm import load
    from mlx_vlm.utils import load_config

    MODEL_ID = "mlx-community/Qwen2-VL-7B-Instruct-4bit"

    print(f"Loading {MODEL_ID} via MLX...")
    model, processor = load(MODEL_ID)
    mlx_config = load_config(MODEL_ID)
    print("Model loaded (4-bit MLX, ~4 GB resident).")
else:
    import torch
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
        Qwen2VLForConditionalGeneration,
    )

    MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    print(f"Loading {MODEL_ID} (4-bit quantized)...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    used = torch.cuda.memory_allocated() / 1e9
    print(f"Model loaded. GPU memory used: {used:.1f} GB")

## Step 3 — OCR scanned PDFs with the GPU

Renders each page to an image and runs it through the vision model. Uses a targeted prompt for SC-100 claim documents that skips bilingual boilerplate; everything else gets the generic extraction prompt. Pages with no case data are dropped.

In [21]:
import io
import time

import fitz
from PIL import Image

GENERIC_PROMPT = (
    "This is a page from a San Francisco small claims court document. "
    "Extract all text exactly as it appears. Include party names, dates, "
    "claim amounts, rulings, and any other information on the page. "
    "Output plain text only."
)

CLAIM_PROMPT = (
    "This is a page from a California SC-100 small claims complaint form. "
    "SKIP all boilerplate instructions, bilingual notices, and form guidance. "
    "Extract ONLY the filled-in case-specific information:\n"
    "- Plaintiff name(s), address, phone\n"
    "- Defendant name(s), address, phone\n"
    "- Trial date and department\n"
    "- Dollar amount claimed and basis for the amount\n"
    "- Description of what happened (the plaintiff's narrative)\n"
    "- Any evidence, witnesses, or documents referenced\n"
    "- Whether plaintiff tried to resolve before suing\n\n"
    "If a page is only boilerplate instructions with no filled-in data, "
    "respond with: [NO CASE DATA ON THIS PAGE]\n"
    "Output plain text only."
)
DPI = 150


def get_prompt_for_doc(filename):
    name_upper = filename.upper()
    if "CLAIM_OF_PLAINTIFF" in name_upper or "DEFENDANT_S_CLAIM" in name_upper:
        return CLAIM_PROMPT
    return GENERIC_PROMPT


def page_to_pil(page, dpi=DPI):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def extract_page_mlx(pil_image, prompt):
    from mlx_vlm import generate
    from mlx_vlm.prompt_utils import apply_chat_template

    formatted = apply_chat_template(processor, mlx_config, prompt, num_images=1)
    output = generate(
        model,
        processor,
        formatted,
        image=[pil_image],
        max_tokens=2048,
        temp=0.0,
        verbose=False,
    )
    # mlx-vlm.generate returns either a string or a GenerationResult; normalize.
    return output if isinstance(output, str) else getattr(output, "text", str(output))


def extract_page_cuda(pil_image, prompt):
    import torch
    from qwen_vl_utils import process_vision_info

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=2048, do_sample=False)

    generated = output_ids[0, inputs.input_ids.shape[1] :]
    return processor.decode(generated, skip_special_tokens=True)


extract_page = extract_page_mlx if BACKEND == "mlx" else extract_page_cuda


if needs_gpu and model is not None:
    t0 = time.time()
    skipped_corrupt = []
    for i, pdf_path in enumerate(needs_gpu):
        txt_path = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"

        if txt_path.exists():
            continue

        prompt = get_prompt_for_doc(pdf_path.name)
        try:
            doc = fitz.open(str(pdf_path))
        except Exception as e:
            # Partial downloads / HTML error pages from a stale SessionID land here
            # (FzErrorFormat "no objects found"). Log + skip; .unlink() and re-run
            # local_pdf_pipeline.ipynb to re-download.
            print(f"  [{i + 1}/{len(needs_gpu)}] SKIP corrupt PDF: {pdf_path.name} ({e})")
            skipped_corrupt.append(pdf_path)
            continue

        pages_text = []

        for page_num, page in enumerate(doc, start=1):
            print(
                f"  [{i + 1}/{len(needs_gpu)}] {pdf_path.name} - page {page_num}/{len(doc)}",
                flush=True,
            )
            img = page_to_pil(page)
            text = extract_page(img, prompt)
            if "[NO CASE DATA ON THIS PAGE]" not in text:
                pages_text.append(text)

        doc.close()
        if BACKEND == "cuda":
            import torch
            torch.cuda.empty_cache()

        full_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"  Saved: {txt_path.name} ({len(full_text):,} chars)")

    elapsed = time.time() - t0
    print(f"\nGPU extraction done: {len(needs_gpu)} PDFs in {elapsed:.0f}s")
    if skipped_corrupt:
        print(f"\nSkipped {len(skipped_corrupt)} corrupt PDFs (re-download via local_pdf_pipeline.ipynb):")
        for p in skipped_corrupt:
            print(f"  {p.name}")
else:
    print("No scanned PDFs to process.")

In [22]:
# Sync results back to Drive (RunPod only — Mac and Colab write directly through their mounts).
# Run this after the OCR loop and again after Steps 4-5 (or any time you want to push state).
# rclone copy is incremental: only changed/new files are uploaded.
if IS_GENERIC_CUDA:
    import subprocess
    print("Syncing /workspace/litigation-pipeline → drive:litigation-pipeline ...")
    subprocess.run(
        [
            "rclone", "copy", "/workspace/litigation-pipeline",
            "drive:litigation-pipeline",
            "--transfers", "16",
            "--checkers", "32",
            "--progress",
        ],
        check=True,
    )
    print("Sync complete — teammates' Drive folder is up to date.")
else:
    print("Skipping — Mac/Colab write directly through their Drive mount.")

## Step 4 — Extract labels with GPT-4o-mini

Reads outcome documents (judgments, orders, dismissals) and extracts structured labels: win/loss/dismissed, amount awarded, whether defendant appeared, etc.

Requires an OpenAI API key (set `LLM_API_KEY` below). Costs ~$0.001 per case.

In [23]:
import json

LLM_API_KEY = input("OpenAI API key (leave blank to skip label extraction): ").strip()

if LLM_API_KEY:
    from features.config import FeaturesConfig
    from features.labels import LabelExtractor

    label_config = FeaturesConfig(
        llm_api_key=LLM_API_KEY,
        llm_model="gpt-4o-mini",
        cache_dir=f"{DRIVE_DIR}/labels_cache",
    )
    extractor = LabelExtractor(config=label_config)

    txt_dir = Path(OUTPUT_DIR)
    case_numbers = sorted(
        {f.name.split("_")[0] for f in txt_dir.glob("*.txt") if f.name.startswith("CSM")}
    )
    print(f"Found {len(case_numbers)} cases to label")

    all_labels = {}
    for i, case_num in enumerate(case_numbers):
        labels = extractor.extract_labels(case_num, txt_dir)
        if labels:
            all_labels[case_num] = labels.model_dump()
            print(
                f"  [{i + 1}/{len(case_numbers)}] {case_num}: {labels.outcome} "
                f"(${labels.total_awarded or 0:.2f})"
            )

    labels_path = Path(DRIVE_DIR) / "labels.json"
    labels_path.parent.mkdir(parents=True, exist_ok=True)
    labels_path.write_text(json.dumps(all_labels, indent=2), encoding="utf-8")
    print(f"\nLabels saved: {labels_path} ({len(all_labels)} cases)")
else:
    print("Skipping label extraction (no API key).")

## Step 5 — Extract features with GPT-4o-mini

Builds a `ProcessedCase` per case (excluding label docs to preserve the leakage firewall) and runs `FeatureExtractor` to produce the v2 existence-based feature vectors. Cache lands in Drive so progress survives runtime restarts.

Reuses the OpenAI API key entered in Step 4; prompts again if you skipped it. Costs ~$0.001 per case.

In [24]:
from datetime import date

from data.schemas.case import ExtractedText, ProcessedCase
from features.config import FeaturesConfig
from features.extraction import FeatureExtractor
from features.labels import _is_label_doc


# Per-case ProcessedCase builder. Diverges from
# scripts/extract_features_from_local_cases.py::_build_processed_cases because
# this version takes an arbitrary txt_dir (Drive-mounted) and supports per-case
# user_side detection + document_texts inclusion.
def build_processed_case(case_number, txt_dir):
    all_txts = sorted(txt_dir.glob(f"{case_number}_*.txt"))
    feature_txts = [p for p in all_txts if not _is_label_doc(p.name)]
    if not feature_txts:
        return None

    has_defendant_claim = any("DEFENDANT_S_CLAIM" in p.name.upper() for p in all_txts)
    user_side = "defendant" if has_defendant_claim else "plaintiff"

    document_texts = []
    text_parts = []
    for p in feature_txts:
        text = p.read_text(encoding="utf-8").strip()
        if not text:
            continue
        document_texts.append(
            ExtractedText(
                case_number=case_number,
                document_filename=p.stem,
                pages=[text],
            )
        )
        text_parts.append(text)

    if not text_parts:
        return None

    return ProcessedCase(
        case_number=case_number,
        case_title="",
        cause_of_action=None,
        filing_date=date(1900, 1, 1),
        full_text="\n\n---\n\n".join(text_parts),
        document_texts=document_texts,
        claim_amount=None,
        has_attorney_plaintiff=None,
        has_attorney_defendant=None,
        user_side=user_side,
    )


# Reuse the API key from Step 4 if available; re-prompt if missing.
LLM_API_KEY = globals().get("LLM_API_KEY") or input(
    "OpenAI API key (leave blank to skip feature extraction): "
).strip()

if LLM_API_KEY:
    feature_config = FeaturesConfig(
        llm_api_key=LLM_API_KEY,
        llm_model="gpt-4o-mini",
        cache_dir=f"{DRIVE_DIR}/features_cache",
    )
    feature_extractor = FeatureExtractor(config=feature_config)

    txt_dir = Path(OUTPUT_DIR)
    case_numbers = sorted(
        {f.name.split("_")[0] for f in txt_dir.glob("*.txt") if f.name.startswith("CSM")}
    )

    if TOTAL_WORKERS > 1:
        case_numbers = case_numbers[MY_WORKER::TOTAL_WORKERS]

    print(f"Found {len(case_numbers)} cases in this worker's slice")

    processed_cases = []
    for case_num in case_numbers:
        pc = build_processed_case(case_num, txt_dir)
        if pc is not None:
            processed_cases.append(pc)

    print(f"Built {len(processed_cases)} ProcessedCase objects; running extraction...")

    feature_vectors = await feature_extractor.extract_batch(processed_cases)

    print(f"\nFeature extraction complete: {len(feature_vectors)} / {len(processed_cases)} cases")
    print(f"Cache dir: {feature_config.cache_dir}")
else:
    print("Skipping feature extraction (no API key).")

## Results

Summary of extracted text and labels. Bundles everything as a zip download.

In [26]:
import shutil

txt_files = sorted(Path(OUTPUT_DIR).glob("*.txt"))
total_chars = sum(f.stat().st_size for f in txt_files)

print(f"Total text files: {len(txt_files)}")
print(f"Total size:       {total_chars:,} characters ({total_chars / 1024:.0f} KB)")
print()

for f in txt_files[:20]:
    print(f"  {f.name}  ({f.stat().st_size:,} chars)")
if len(txt_files) > 20:
    print(f"  ... and {len(txt_files) - 20} more")

labels_path = Path(DRIVE_DIR) / "labels.json"
if labels_path.exists():
    labels = json.loads(labels_path.read_text())
    outcomes = {}
    for v in labels.values():
        o = v.get("outcome", "unknown") or "unknown"
        outcomes[o] = outcomes.get(o, 0) + 1
    print(f"\nLabels: {len(labels)} cases")
    for outcome, count in sorted(outcomes.items()):
        print(f"  {outcome}: {count}")

print(f"\nFiles persist in Google Drive at: {DRIVE_DIR}")

if IS_COLAB:
    zip_path = "/content/extracted_texts"
    shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
    print(f"\nZip: {zip_path}.zip ({Path(zip_path + '.zip').stat().st_size / 1024:.0f} KB)")
    files.download(f"{zip_path}.zip")
else:
    print("\n(Skipping zip + download — outputs already live in the synced Drive folder.)")